# Smallest stiffness eigenvalue versus strain — SIM 5000, 5010, 5100, 5110, and 5510 families

This notebook makes separate plots for SIM_5000–SIM_5009, SIM_5010–SIM_5019, SIM_5100–SIM_5109, SIM_5110–SIM_5119, and SIM_5510–SIM_5519. It loads the heavy `DATA_PICK_*_EIGV.pkl` files, which contain the 401-point eigenvalue histories (the initial state plus 400 increments). Curves are labeled by the internal pressure used to create each simulation.

For this family, Step-1 applies a 5 mm displacement over a 20 mm specimen dimension. The nominal compressive strain is therefore defined as `0.25 * t`, where `t` is the normalized Step-1 time stored in `DATA_PICK_*_EIGV.pkl`.

In [ ]:
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path('../I001_Results')
if not RESULTS_DIR.exists():
    RESULTS_DIR = Path('I001_Results')
PRESSURES = (0.0,) + tuple(0.04 * np.sqrt(2.0) ** i for i in range(9))
FAMILIES = {
    'SIM 5000 family — hexagonal packing, linear material': range(5000, 5010),
    'SIM 5010 family — random packing, linear material': range(5010, 5020),
    'SIM 5100 family — random packing, neo-Hookean material': range(5100, 5110),
    'SIM 5110 family — random packing, neo-Hookean material': range(5110, 5120),
    'SIM 5510 family': range(5510, 5520),
}
EXPECTED_EIGENVALUE_POINTS = 401

# The 5000-series EIGV pickles are the heavy files with 401 snapshots: the initial state plus 400 increments.
SPECIMEN_DIMENSION = 20.0  # mm
IMPOSED_DISPLACEMENT = 5.0  # mm
MAX_STRAIN = IMPOSED_DISPLACEMENT / SPECIMEN_DIMENSION

print(f'Result directory: {RESULTS_DIR.resolve()}')
print(f'Nominal maximum compressive strain: {MAX_STRAIN:.1%}')


In [ ]:
curves = {}
missing = []
invalid = []

for family_name, sim_ids in FAMILIES.items():
    curves[family_name] = {}
    for sim_id in sim_ids:
        path = RESULTS_DIR / f'DATA_PICK_{sim_id}_EIGV.pkl'
        if not path.exists():
            missing.append(sim_id)
            continue

        with path.open('rb') as handle:
            data = pickle.load(handle)

        try:
            time = np.asarray(data['t'], dtype=float)
            eigenvalues = np.asarray(data['eigenvalues'], dtype=float)
            if eigenvalues.ndim != 2 or eigenvalues.shape[1] == 0:
                raise ValueError('eigenvalues does not contain mode columns')
            if len(time) != len(eigenvalues):
                raise ValueError('time and eigenvalue lengths differ')
            if len(time) != EXPECTED_EIGENVALUE_POINTS:
                raise ValueError(
                    f'expected {EXPECTED_EIGENVALUE_POINTS} eigenvalue points, '
                    f'found {len(time)}'
                )
        except (KeyError, TypeError, ValueError) as error:
            invalid.append((sim_id, str(error)))
            continue

        # The first stored mode is the smallest eigenvalue at each snapshot.
        strain = MAX_STRAIN * time
        pressure = PRESSURES[sim_id % 10]
        # Locate the first positive-to-negative crossing of the smallest eigenvalue.
        crossing = np.flatnonzero((eigenvalues[:-1, 0] > 0.0) & (eigenvalues[1:, 0] <= 0.0))
        if len(crossing):
            i = int(crossing[0])
            t0, t1 = time[i:i + 2]
            l0, l1 = eigenvalues[i:i + 2, 0]
            crossing_t = t0 - l0 * (t1 - t0) / (l1 - l0)
        else:
            crossing_t = np.nan

        # Copy the one required mode so the multi-GB eigenvector payload can be released.
        lambda_min = eigenvalues[:, 0].copy()

        # f is the ratio of the average global tension and compression efficiencies.
        f_at_crossing = np.nan
        i3_path = RESULTS_DIR / f'DATA_PICK_{sim_id}_I3_BFS_3002.pkl'
        if i3_path.exists() and np.isfinite(crossing_t):
            with i3_path.open('rb') as handle:
                i3 = pickle.load(handle)
            i3_time = np.asarray(i3['t'], dtype=float)
            ef_t = np.asarray(i3['global_ef_t'], dtype=float)
            ef_c = np.asarray(i3['global_ef_c'], dtype=float)
            if len(i3_time) == len(time) == len(ef_t) == len(ef_c):
                ef_t_crossing = np.interp(crossing_t, i3_time, ef_t)
                ef_c_crossing = np.interp(crossing_t, i3_time, ef_c)
                if ef_c_crossing != 0.0:
                    f_at_crossing = ef_t_crossing / ef_c_crossing
        curves[family_name][sim_id] = {
            'time': time,
            'strain': strain,
            'lambda_min': lambda_min,
            'pressure': pressure,
            'crossing_t': crossing_t,
            'f_at_crossing': f_at_crossing,
        }

print('Loaded curves:')
for family_name, family_curves in curves.items():
    print(f'  {family_name}: {sorted(family_curves)}')
if missing:
    print(f'Missing eigenvalue files for: {missing}')
if invalid:
    print(f'Invalid eigenvalue files: {invalid}')


In [ ]:
figures = {}
for family_name, family_curves in curves.items():
    fig, ax = plt.subplots(figsize=(11, 7), constrained_layout=True)
    for sim_id, data in sorted(family_curves.items()):
        line, = ax.plot(100 * data['strain'], data['lambda_min'], linewidth=1.5, label=f"p = {data['pressure']:.5g}")
        if np.isfinite(data['crossing_t']):
            ax.scatter(100 * MAX_STRAIN * data['crossing_t'], 0.0, color=line.get_color(),
                       edgecolor='black', s=42, zorder=5)
    ax.axhline(0.0, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Nominal compressive strain (%)')
    ax.set_ylabel('Smallest tangent-stiffness eigenvalue')
    ax.set_title(f'{family_name}: smallest eigenvalue versus strain')
    ax.grid(True, alpha=0.3)
    if family_curves:
        ax.legend(title='Internal pressure', fontsize=9, loc='best')
    figures[family_name] = fig
    plt.show()

# Requested 5000-family summary: compression ratio (normalized Step-1 time)
# at the first zero crossing versus the internal pressure.
family_name = 'SIM 5000 family — hexagonal packing, linear material'
crossing_rows = []
for sim_id, data in sorted(curves[family_name].items()):
    if np.isfinite(data['crossing_t']):
        crossing_rows.append((sim_id, data['pressure'], data['crossing_t']))

crossing_rows = np.asarray(crossing_rows, dtype=float)
if crossing_rows.size:
    fig_crossing, ax_crossing = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_crossing.plot(crossing_rows[:, 1], crossing_rows[:, 2], 'o-', linewidth=1.5)
    for sim_id, pressure, compression_ratio in crossing_rows:
        ax_crossing.annotate(f'SIM {int(sim_id)}', (pressure, compression_ratio),
                             xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_crossing.set_xlabel('Internal pressure, p')
    ax_crossing.set_ylabel('Compression ratio at first eigenvalue zero crossing')
    ax_crossing.set_title('SIM 5000 family: first eigenvalue zero crossing')
    ax_crossing.grid(True, alpha=0.3)
    plt.show()
else:
    fig_crossing = None
    print('No zero crossing was found for the available SIM 5000 curves.')

# f evaluated at the same interpolated zero-crossing points.
f_rows = [(sim_id, data['pressure'], data['crossing_t'], data['f_at_crossing'])
           for sim_id, data in sorted(curves[family_name].items())
           if np.isfinite(data['crossing_t']) and np.isfinite(data['f_at_crossing'])]
f_rows = np.asarray(f_rows, dtype=float)
missing_f = [sim_id for sim_id, data in sorted(curves[family_name].items())
             if np.isfinite(data['crossing_t']) and not np.isfinite(data['f_at_crossing'])]
if missing_f:
    print(f'f could not be evaluated at the zero crossing for: {missing_f} '
          '(matching I3 efficiency files are unavailable).')
if f_rows.size:
    print('f at first eigenvalue zero crossing:')
    print('  SIM       pressure   compression ratio       f')
    for sim_id, pressure, compression_ratio, f_value in f_rows:
        print(f'  {int(sim_id):4d}   {pressure:10.6g}   {compression_ratio:17.8f}   {f_value: .8f}')

    fig_f_pressure, ax_f_pressure = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_f_pressure.plot(f_rows[:, 1], f_rows[:, 3], 'o-', linewidth=1.5)
    for sim_id, pressure, _, f_value in f_rows:
        ax_f_pressure.annotate(f'SIM {int(sim_id)}', (pressure, f_value),
                              xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_f_pressure.set_xlabel('Internal pressure, p')
    ax_f_pressure.set_ylabel('f = average global tension efficiency / compression efficiency')
    ax_f_pressure.set_title('SIM 5000 family: f at first eigenvalue zero crossing vs pressure')
    ax_f_pressure.grid(True, alpha=0.3)
    plt.show()

    fig_f_ratio, ax_f_ratio = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_f_ratio.plot(f_rows[:, 2], f_rows[:, 3], 'o-', linewidth=1.5)
    for sim_id, compression_ratio, f_value in zip(f_rows[:, 0], f_rows[:, 2], f_rows[:, 3]):
        ax_f_ratio.annotate(f'SIM {int(sim_id)}', (compression_ratio, f_value),
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_f_ratio.set_xlabel('Compression ratio at first eigenvalue zero crossing')
    ax_f_ratio.set_ylabel('f = average global tension efficiency / compression efficiency')
    ax_f_ratio.set_title('SIM 5000 family: f at first eigenvalue zero crossing vs compression ratio')
    ax_f_ratio.grid(True, alpha=0.3)
    plt.show()
else:
    fig_f_pressure = None
    fig_f_ratio = None
    print('No valid f values were found at the available zero crossings.')

# Repeat the 5000-family summary plots for the SIM 5010 family.
family_name = 'SIM 5010 family — random packing, linear material'
crossing_rows_5010 = [(sim_id, data['pressure'], data['crossing_t'])
                     for sim_id, data in sorted(curves[family_name].items())
                     if np.isfinite(data['crossing_t'])]
crossing_rows_5010 = np.asarray(crossing_rows_5010, dtype=float)
if crossing_rows_5010.size:
    fig_crossing_5010, ax_crossing_5010 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_crossing_5010.plot(crossing_rows_5010[:, 1], crossing_rows_5010[:, 2], 'o-', linewidth=1.5)
    for sim_id, pressure, compression_ratio in crossing_rows_5010:
        ax_crossing_5010.annotate(f'SIM {int(sim_id)}', (pressure, compression_ratio),
                                 xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_crossing_5010.set_xlabel('Internal pressure, p')
    ax_crossing_5010.set_ylabel('Compression ratio at first eigenvalue zero crossing')
    ax_crossing_5010.set_title('SIM 5010 family: first eigenvalue zero crossing')
    ax_crossing_5010.grid(True, alpha=0.3)
    plt.show()
else:
    fig_crossing_5010 = None
    print('No zero crossing was found for the available SIM 5010 curves.')

f_rows_5010 = [(sim_id, data['pressure'], data['crossing_t'], data['f_at_crossing'])
              for sim_id, data in sorted(curves[family_name].items())
              if np.isfinite(data['crossing_t']) and np.isfinite(data['f_at_crossing'])]
f_rows_5010 = np.asarray(f_rows_5010, dtype=float)
missing_f_5010 = [sim_id for sim_id, data in sorted(curves[family_name].items())
                  if np.isfinite(data['crossing_t']) and not np.isfinite(data['f_at_crossing'])]
if missing_f_5010:
    print(f'f could not be evaluated at the zero crossing for: {missing_f_5010} '
          '(matching I3 efficiency files are unavailable).')
if f_rows_5010.size:
    print('f at first eigenvalue zero crossing for SIM 5010:')
    print('  SIM       pressure   compression ratio       f')
    for sim_id, pressure, compression_ratio, f_value in f_rows_5010:
        print(f'  {int(sim_id):4d}   {pressure:10.6g}   {compression_ratio:17.8f}   {f_value: .8f}')

    fig_f_pressure_5010, ax_f_pressure_5010 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_f_pressure_5010.plot(f_rows_5010[:, 1], f_rows_5010[:, 3], 'o-', linewidth=1.5)
    for sim_id, pressure, _, f_value in f_rows_5010:
        ax_f_pressure_5010.annotate(f'SIM {int(sim_id)}', (pressure, f_value),
                                   xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_f_pressure_5010.set_xlabel('Internal pressure, p')
    ax_f_pressure_5010.set_ylabel('f = average global tension efficiency / compression efficiency')
    ax_f_pressure_5010.set_title('SIM 5010 family: f at first eigenvalue zero crossing vs pressure')
    ax_f_pressure_5010.grid(True, alpha=0.3)
    plt.show()

    fig_f_ratio_5010, ax_f_ratio_5010 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_f_ratio_5010.plot(f_rows_5010[:, 2], f_rows_5010[:, 3], 'o-', linewidth=1.5)
    for sim_id, compression_ratio, f_value in zip(f_rows_5010[:, 0], f_rows_5010[:, 2], f_rows_5010[:, 3]):
        ax_f_ratio_5010.annotate(f'SIM {int(sim_id)}', (compression_ratio, f_value),
                              xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_f_ratio_5010.set_xlabel('Compression ratio at first eigenvalue zero crossing')
    ax_f_ratio_5010.set_ylabel('f = average global tension efficiency / compression efficiency')
    ax_f_ratio_5010.set_title('SIM 5010 family: f at first eigenvalue zero crossing vs compression ratio')
    ax_f_ratio_5010.grid(True, alpha=0.3)
    plt.show()
else:
    fig_f_pressure_5010 = None
    fig_f_ratio_5010 = None
    print('No valid f values were found at the available SIM 5010 zero crossings.')

# Repeat the 5000-family summary plots for the SIM 5100 family.
family_name = 'SIM 5100 family — random packing, neo-Hookean material'
crossing_rows_5100 = [(sim_id, data['pressure'], data['crossing_t'])
                     for sim_id, data in sorted(curves[family_name].items())
                     if np.isfinite(data['crossing_t'])]
crossing_rows_5100 = np.asarray(crossing_rows_5100, dtype=float)
if crossing_rows_5100.size:
    fig_crossing_5100, ax_crossing_5100 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_crossing_5100.plot(crossing_rows_5100[:, 1], crossing_rows_5100[:, 2], 'o-', linewidth=1.5)
    for sim_id, pressure, compression_ratio in crossing_rows_5100:
        ax_crossing_5100.annotate(f'SIM {int(sim_id)}', (pressure, compression_ratio),
                                 xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_crossing_5100.set_xlabel('Internal pressure, p')
    ax_crossing_5100.set_ylabel('Compression ratio at first eigenvalue zero crossing')
    ax_crossing_5100.set_title('SIM 5100 family: first eigenvalue zero crossing')
    ax_crossing_5100.grid(True, alpha=0.3)
    plt.show()
else:
    fig_crossing_5100 = None
    print('No zero crossing was found for the available SIM 5100 curves.')

f_rows_5100 = [(sim_id, data['pressure'], data['crossing_t'], data['f_at_crossing'])
              for sim_id, data in sorted(curves[family_name].items())
              if np.isfinite(data['crossing_t']) and np.isfinite(data['f_at_crossing'])]
f_rows_5100 = np.asarray(f_rows_5100, dtype=float)
missing_f_5100 = [sim_id for sim_id, data in sorted(curves[family_name].items())
                  if np.isfinite(data['crossing_t']) and not np.isfinite(data['f_at_crossing'])]
if missing_f_5100:
    print(f'f could not be evaluated at the zero crossing for: {missing_f_5100} '
          '(matching I3 efficiency files are unavailable).')
if f_rows_5100.size:
    print('f at first eigenvalue zero crossing for SIM 5100:')
    print('  SIM       pressure   compression ratio       f')
    for sim_id, pressure, compression_ratio, f_value in f_rows_5100:
        print(f'  {int(sim_id):4d}   {pressure:10.6g}   {compression_ratio:17.8f}   {f_value: .8f}')

    fig_f_pressure_5100, ax_f_pressure_5100 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_f_pressure_5100.plot(f_rows_5100[:, 1], f_rows_5100[:, 3], 'o-', linewidth=1.5)
    for sim_id, pressure, _, f_value in f_rows_5100:
        ax_f_pressure_5100.annotate(f'SIM {int(sim_id)}', (pressure, f_value),
                                   xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_f_pressure_5100.set_xlabel('Internal pressure, p')
    ax_f_pressure_5100.set_ylabel('f = average global tension efficiency / compression efficiency')
    ax_f_pressure_5100.set_title('SIM 5100 family: f at first eigenvalue zero crossing vs pressure')
    ax_f_pressure_5100.grid(True, alpha=0.3)
    plt.show()

    fig_f_ratio_5100, ax_f_ratio_5100 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_f_ratio_5100.plot(f_rows_5100[:, 2], f_rows_5100[:, 3], 'o-', linewidth=1.5)
    for sim_id, compression_ratio, f_value in zip(f_rows_5100[:, 0], f_rows_5100[:, 2], f_rows_5100[:, 3]):
        ax_f_ratio_5100.annotate(f'SIM {int(sim_id)}', (compression_ratio, f_value),
                              xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_f_ratio_5100.set_xlabel('Compression ratio at first eigenvalue zero crossing')
    ax_f_ratio_5100.set_ylabel('f = average global tension efficiency / compression efficiency')
    ax_f_ratio_5100.set_title('SIM 5100 family: f at first eigenvalue zero crossing vs compression ratio')
    ax_f_ratio_5100.grid(True, alpha=0.3)
    plt.show()
else:
    fig_f_pressure_5100 = None
    fig_f_ratio_5100 = None
    print('No valid f values were found at the available SIM 5100 zero crossings.')

# Repeat the 5000-family summary plots for the SIM 5510 family.
family_name = 'SIM 5510 family'
crossing_rows_5510 = [(sim_id, data['pressure'], data['crossing_t'])
                     for sim_id, data in sorted(curves[family_name].items())
                     if np.isfinite(data['crossing_t'])]
crossing_rows_5510 = np.asarray(crossing_rows_5510, dtype=float)
if crossing_rows_5510.size:
    fig_crossing_5510, ax_crossing_5510 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_crossing_5510.plot(crossing_rows_5510[:, 1], crossing_rows_5510[:, 2], 'o-', linewidth=1.5)
    for sim_id, pressure, compression_ratio in crossing_rows_5510:
        ax_crossing_5510.annotate(f'SIM {int(sim_id)}', (pressure, compression_ratio),
                                 xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_crossing_5510.set_xlabel('Internal pressure, p')
    ax_crossing_5510.set_ylabel('Compression ratio at first eigenvalue zero crossing')
    ax_crossing_5510.set_title('SIM 5510 family: first eigenvalue zero crossing')
    ax_crossing_5510.grid(True, alpha=0.3)
    plt.show()
else:
    fig_crossing_5510 = None
    print('No zero crossing was found for the available SIM 5510 curves.')

f_rows_5510 = [(sim_id, data['pressure'], data['crossing_t'], data['f_at_crossing'])
              for sim_id, data in sorted(curves[family_name].items())
              if np.isfinite(data['crossing_t']) and np.isfinite(data['f_at_crossing'])]
f_rows_5510 = np.asarray(f_rows_5510, dtype=float)
if f_rows_5510.size:
    print('f at first eigenvalue zero crossing for SIM 5510:')
    print('  SIM       pressure   compression ratio       f')
    for sim_id, pressure, compression_ratio, f_value in f_rows_5510:
        print(f'  {int(sim_id):4d}   {pressure:10.6g}   {compression_ratio:17.8f}   {f_value: .8f}')

    fig_f_pressure_5510, ax_f_pressure_5510 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_f_pressure_5510.plot(f_rows_5510[:, 1], f_rows_5510[:, 3], 'o-', linewidth=1.5)
    for sim_id, pressure, _, f_value in f_rows_5510:
        ax_f_pressure_5510.annotate(f'SIM {int(sim_id)}', (pressure, f_value),
                                   xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_f_pressure_5510.set_xlabel('Internal pressure, p')
    ax_f_pressure_5510.set_ylabel('f = average global tension efficiency / compression efficiency')
    ax_f_pressure_5510.set_title('SIM 5510 family: f at first eigenvalue zero crossing vs pressure')
    ax_f_pressure_5510.grid(True, alpha=0.3)
    plt.show()

    fig_f_ratio_5510, ax_f_ratio_5510 = plt.subplots(figsize=(9, 6), constrained_layout=True)
    ax_f_ratio_5510.plot(f_rows_5510[:, 2], f_rows_5510[:, 3], 'o-', linewidth=1.5)
    for sim_id, compression_ratio, f_value in zip(f_rows_5510[:, 0], f_rows_5510[:, 2], f_rows_5510[:, 3]):
        ax_f_ratio_5510.annotate(f'SIM {int(sim_id)}', (compression_ratio, f_value),
                              xytext=(5, 5), textcoords='offset points', fontsize=8)
    ax_f_ratio_5510.set_xlabel('Compression ratio at first eigenvalue zero crossing')
    ax_f_ratio_5510.set_ylabel('f = average global tension efficiency / compression efficiency')
    ax_f_ratio_5510.set_title('SIM 5510 family: f at first eigenvalue zero crossing vs compression ratio')
    ax_f_ratio_5510.grid(True, alpha=0.3)
    plt.show()
else:
    fig_f_pressure_5510 = None
    fig_f_ratio_5510 = None
    print('No valid f values were found at the available SIM 5510 zero crossings.')


In [ ]:
# Optional: save all figures next to the notebook.
output_paths = {
    'SIM 5000 family — hexagonal packing, linear material': Path('SIM_5000s_smallest_eigenvalue_vs_strain.png'),
    'SIM 5010 family — random packing, linear material': Path('SIM_5010s_smallest_eigenvalue_vs_strain.png'),
    'SIM 5100 family — random packing, neo-Hookean material': Path('SIM_5100s_smallest_eigenvalue_vs_strain.png'),
    'SIM 5110 family — random packing, neo-Hookean material': Path('SIM_5110s_smallest_eigenvalue_vs_strain.png'),
    'SIM 5510 family': Path('SIM_5510s_smallest_eigenvalue_vs_strain.png'),
}
for family_name, output_path in output_paths.items():
    if family_name in figures:
        figures[family_name].savefig(output_path, dpi=200, bbox_inches='tight')
        print(f'Saved: {output_path.resolve()}')
if fig_crossing is not None:
    crossing_output = Path('SIM_5000s_first_eigenvalue_zero_crossing_vs_pressure.png')
    fig_crossing.savefig(crossing_output, dpi=200, bbox_inches='tight')
    print(f'Saved: {crossing_output.resolve()}')
if fig_f_pressure is not None:
    f_pressure_output = Path('SIM_5000s_f_at_eigenvalue_zero_vs_pressure.png')
    fig_f_pressure.savefig(f_pressure_output, dpi=200, bbox_inches='tight')
    print(f'Saved: {f_pressure_output.resolve()}')
if fig_f_ratio is not None:
    f_ratio_output = Path('SIM_5000s_f_at_eigenvalue_zero_vs_compression_ratio.png')
    fig_f_ratio.savefig(f_ratio_output, dpi=200, bbox_inches='tight')
    print(f'Saved: {f_ratio_output.resolve()}')
if fig_crossing_5010 is not None:
    crossing_output_5010 = Path('SIM_5010s_first_eigenvalue_zero_crossing_vs_pressure.png')
    fig_crossing_5010.savefig(crossing_output_5010, dpi=200, bbox_inches='tight')
    print(f'Saved: {crossing_output_5010.resolve()}')
if fig_f_pressure_5010 is not None:
    f_pressure_output_5010 = Path('SIM_5010s_f_at_eigenvalue_zero_vs_pressure.png')
    fig_f_pressure_5010.savefig(f_pressure_output_5010, dpi=200, bbox_inches='tight')
    print(f'Saved: {f_pressure_output_5010.resolve()}')
if fig_f_ratio_5010 is not None:
    f_ratio_output_5010 = Path('SIM_5010s_f_at_eigenvalue_zero_vs_compression_ratio.png')
    fig_f_ratio_5010.savefig(f_ratio_output_5010, dpi=200, bbox_inches='tight')
    print(f'Saved: {f_ratio_output_5010.resolve()}')
if fig_crossing_5100 is not None:
    crossing_output_5100 = Path('SIM_5100s_first_eigenvalue_zero_crossing_vs_pressure.png')
    fig_crossing_5100.savefig(crossing_output_5100, dpi=200, bbox_inches='tight')
    print(f'Saved: {crossing_output_5100.resolve()}')
if fig_f_pressure_5100 is not None:
    f_pressure_output_5100 = Path('SIM_5100s_f_at_eigenvalue_zero_vs_pressure.png')
    fig_f_pressure_5100.savefig(f_pressure_output_5100, dpi=200, bbox_inches='tight')
    print(f'Saved: {f_pressure_output_5100.resolve()}')
if fig_f_ratio_5100 is not None:
    f_ratio_output_5100 = Path('SIM_5100s_f_at_eigenvalue_zero_vs_compression_ratio.png')
    fig_f_ratio_5100.savefig(f_ratio_output_5100, dpi=200, bbox_inches='tight')
    print(f'Saved: {f_ratio_output_5100.resolve()}')
if fig_crossing_5510 is not None:
    crossing_output_5510 = Path('SIM_5510s_first_eigenvalue_zero_crossing_vs_pressure.png')
    fig_crossing_5510.savefig(crossing_output_5510, dpi=200, bbox_inches='tight')
    print(f'Saved: {crossing_output_5510.resolve()}')
if fig_f_pressure_5510 is not None:
    f_pressure_output_5510 = Path('SIM_5510s_f_at_eigenvalue_zero_vs_pressure.png')
    fig_f_pressure_5510.savefig(f_pressure_output_5510, dpi=200, bbox_inches='tight')
    print(f'Saved: {f_pressure_output_5510.resolve()}')
if fig_f_ratio_5510 is not None:
    f_ratio_output_5510 = Path('SIM_5510s_f_at_eigenvalue_zero_vs_compression_ratio.png')
    fig_f_ratio_5510.savefig(f_ratio_output_5510, dpi=200, bbox_inches='tight')
    print(f'Saved: {f_ratio_output_5510.resolve()}')
